# Notebook 2: Financial Interpretability of Path Signatures

This notebook walks through the formal mappings between signature terms
and classical financial phenomena:

| Signature term | Financial phenomenon | Direction |
|---|---|---|
| $S^1_r = X^r_T - X^r_0$ | Momentum (cumulative return) | Positive |
| $A_{\text{lead}_r, \text{lag}_r}$ | Realised variance (QV) | — |
| $A_{r, \sigma} = S^2_{r,\sigma} - S^2_{\sigma,r}$ | Leverage effect | Negative for typical equities |
| $A_{r, v} = S^2_{r,v} - S^2_{v,r}$ | Price-volume lead-lag | Sign identifies who leads |


In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec

from data.synthetic import (
    generate_gbm_with_leverage,
    generate_leadlag_volume_price,
    generate_momentum_regime,
)
from pathsig.signatures import compute_signature, extract_signature_term
from pathsig.augmentations import lead_lag_augmentation, time_augmentation
from pathsig.interpretability import (
    interpret_level1,
    interpret_level2_diagonal,
    interpret_level2_cross,
    interpret_price_volume_leadlag,
    full_interpretability_report,
)

plt.rcParams['figure.dpi'] = 120
plt.rcParams['font.size'] = 11
print('Modules loaded.')

## 1. Level-1 → Momentum

$$S^1_r = \int_0^T dX^r_t = X^r_T - X^r_0 = \text{cumulative log-return}$$

This is *exactly* the momentum signal. No approximation — it IS the cumulative return.

In [ ]:
panel_mom = generate_momentum_regime(n_paths=200, n_steps=252, momentum_strength=0.10, seed=0)

window = 21
sig_mom_list, true_mom_list = [], []

for ticker, grp in panel_mom.groupby('ticker'):
    grp = grp.sort_values('date').reset_index(drop=True)
    rets = grp['log_return'].values.astype(float)
    T = len(rets)
    for t in range(window, T):
        path_w = rets[t-window:t].reshape(-1, 1)
        sig = compute_signature(path_w, depth=1)
        sig_mom_list.append(float(sig[0]))
        true_mom_list.append(float(rets[t-window:t].sum()))

sig_mom  = np.array(sig_mom_list)
true_mom = np.array(true_mom_list)

print(f'Correlation(S¹_r, cumulative_return) = {np.corrcoef(sig_mom, true_mom)[0,1]:.8f}')
print('→ S¹_r IS the cumulative return (perfect correspondence by construction)')

## 2. Lead-Lag → Realised Variance

For the lead-lag augmented path, the Lévy area between lead and lag channels equals the quadratic variation:

$$A_{\text{lead}_r, \text{lag}_r} = S^2_{\text{lead}_r, \text{lag}_r} - S^2_{\text{lag}_r, \text{lead}_r} = \sum_k (\Delta X^r_k)^2 = [X^r, X^r]_T$$

In [ ]:
panel_lev = generate_gbm_with_leverage(n_paths=100, n_steps=252, leverage_corr=-0.7, seed=1)

sig_rv_list, direct_rv_list = [], []
for ticker, grp in panel_lev.groupby('ticker'):
    grp = grp.sort_values('date').reset_index(drop=True)
    rets = grp['log_return'].values.astype(float)
    T = len(rets)
    for t in range(window, T):
        r_w = rets[t-window:t]
        path_w = np.cumsum(r_w).reshape(-1, 1)  # price path
        aug = lead_lag_augmentation(path_w)
        sig = compute_signature(aug, depth=2)
        d_aug = 2
        s_ll = extract_signature_term(sig, d=d_aug, multi_index=(0, 1))
        s_lag = extract_signature_term(sig, d=d_aug, multi_index=(1, 0))
        sig_rv_list.append(s_ll - s_lag)          # Lévy area = QV
        direct_rv_list.append(float(np.sum(r_w**2)))  # direct: sum of squared returns

sig_rv    = np.array(sig_rv_list)
direct_rv = np.array(direct_rv_list)

corr = np.corrcoef(sig_rv, direct_rv)[0, 1]
print(f'Correlation(Lévy area, direct RV) = {corr:.8f}')

fig, ax = plt.subplots(figsize=(7, 5))
idx = np.random.choice(len(sig_rv), size=500, replace=False)
ax.scatter(direct_rv[idx]*1e4, sig_rv[idx]*1e4, alpha=0.3, s=8)
ax.set_xlabel('Direct Realised Variance (×10⁴)')
ax.set_ylabel('Signature Lévy Area (×10⁴)')
ax.set_title(f'Lead-Lag Lévy Area = Realised Variance\n(Pearson r = {corr:.6f})')
lim = max(ax.get_xlim()[1], ax.get_ylim()[1])
ax.plot([0, lim], [0, lim], 'r--', alpha=0.5, label='y=x')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 3. Level-2 Lévy Area → The Leverage Effect

Black (1976) documented that negative equity returns tend to increase volatility:

$$A_{r, \sigma} = S^2_{r,\sigma} - S^2_{\sigma,r} < 0 \;\Leftrightarrow\; \text{negative returns predict higher volatility}$$

In [ ]:
rho_values = [-0.8, -0.5, 0.0, 0.5]
levy_means = []

for rho in rho_values:
    panel = generate_gbm_with_leverage(n_paths=50, n_steps=252, leverage_corr=rho, seed=42)
    levy_list = []
    for ticker, grp in panel.groupby('ticker'):
        grp = grp.sort_values('date').reset_index(drop=True)
        rets = grp['log_return'].values.astype(float)
        vols = grp['realized_vol'].values.astype(float)
        T = len(grp)
        for t in range(window, T):
            path_w = np.column_stack([rets[t-window:t], vols[t-window:t]])
            sig = compute_signature(path_w, depth=2)
            d = 2
            s_rv = sig[d + 0*d + 1]   # S^2_{ret, vol}
            s_vr = sig[d + 1*d + 0]   # S^2_{vol, ret}
            levy_list.append(s_rv - s_vr)
    levy_means.append(np.mean(levy_list))

fig, ax = plt.subplots(figsize=(7, 4))
colors = ['red' if v < 0 else 'steelblue' for v in levy_means]
ax.bar([str(r) for r in rho_values], levy_means, color=colors, alpha=0.7, edgecolor='k')
ax.axhline(0, color='k', linewidth=0.8)
ax.set_xlabel('True leverage correlation ρ')
ax.set_ylabel('Mean Lévy area A_{ret, σ}')
ax.set_title('Signature Lévy Area Tracks the Leverage Effect')
ax.grid(True, alpha=0.3, axis='y')
plt.tight_layout()
plt.show()

print('\nρ → Lévy area (mean over all windows):')
for rho, lv in zip(rho_values, levy_means):
    print(f'  ρ={rho:+.1f} → A_{{ret,σ}} = {lv:.6f}  {'✓ negative' if (rho < 0) == (lv < 0) else '✗ wrong sign'}')

## 4. Price-Volume Lead-Lag

When **volume leads price** (informed trading): $A_{r,v} < 0$  
When **price leads volume** (momentum chasing): $A_{r,v} > 0$

In [ ]:
panel_lag = generate_leadlag_volume_price(n_paths=100, n_steps=252, lag=1, seed=5)

levy_pv_list = []
for ticker, grp in panel_lag.groupby('ticker'):
    grp = grp.sort_values('date').reset_index(drop=True)
    rets = grp['log_return'].values.astype(float)
    lvol = grp['log_volume'].values.astype(float)
    T = len(grp)
    for t in range(window, T):
        path_w = np.column_stack([rets[t-window:t], lvol[t-window:t]])
        sig = compute_signature(path_w, depth=2)
        d = 2
        s_pv = sig[d + 0*d + 1]
        s_vp = sig[d + 1*d + 0]
        levy_pv_list.append(s_pv - s_vp)

levy_pv = np.array(levy_pv_list)
detected = 'volume leads price' if levy_pv.mean() < 0 else 'price leads volume'

print(f'True structure: volume leads price (lag=1)')
print(f'Lévy area mean: {levy_pv.mean():.6f} ({"< 0 → volume leads" if levy_pv.mean() < 0 else "> 0 → price leads"})')
print(f'Detected: {detected}  {"✓ CORRECT" if detected == "volume leads price" else "✗ WRONG"}')

fig, ax = plt.subplots(figsize=(7, 4))
ax.hist(levy_pv, bins=60, edgecolor='k', alpha=0.7, color='steelblue')
ax.axvline(0, color='k', linestyle='--', linewidth=1.2, label='zero')
ax.axvline(levy_pv.mean(), color='red', linewidth=2, label=f'mean={levy_pv.mean():.5f}')
ax.set_xlabel('Lévy area A_{price, volume}')
ax.set_title('Distribution of Price-Volume Lévy Area\n(volume leads price → mean < 0)')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 5. Full Interpretability Report

In [ ]:
# Generate a single 30-day window for the full report
panel_full = generate_gbm_with_leverage(n_paths=1, n_steps=60, leverage_corr=-0.7, seed=99)
single = panel_full.sort_values('date')
path_w = single[['log_return', 'realized_vol', 'log_volume']].values[:30]

report = full_interpretability_report(
    path_w,
    depth=2,
    channel_names=['log_return', 'realized_vol', 'log_volume'],
    augmentations=['lead_lag'],
    price_channel=0,
    vol_channel=1,
    volume_channel=2,
)

print('=== FULL INTERPRETABILITY REPORT ===')
print(f"\nAugmented path shape: {report['augmented_path_shape']}")
print(f"Signature dimension:  {report['signature_dimension']}")

print('\n[Level 1 — Momentum]')
for ch, info in report['level1_momentum'].items():
    print(f"  {ch}: value={info['value']:.6f}  | {info['financial_meaning']}")

print('\n[Level 2 — Realised Variance via Lévy Area]')
for ch, info in report['level2_realized_variance'].items():
    print(f"  {ch}: QV={info['realized_variance']:.8f}, vol={info['realized_volatility']:.6f}")

print('\n[Level 2 — Leverage Effect]')
lev = report['leverage_effect']
print(f"  Lévy area A_{{ret,vol}} = {lev['levy_area']:.6f}")
print(f"  Interpretation: {lev['interpretation']}")

print('\n[Level 2 — Price-Volume Lead-Lag]')
ll = report['price_volume_leadlag']
print(f"  Lévy area A_{{price,vol}} = {ll['levy_area']:.6f}")
print(f"  Direction: {ll['lead_lag_direction']}")